# Downstream HPP Feature Explorer

This notebook inspects the HPP `food_id`-level feature tables prepared for NutriMatch-style prediction tasks inside the TRE.

The notebook is deliberately focused on paper-relevant checks:

- what downstream feature recipes exist;
- how de novo and NutriMatch-based branches differ;
- which feature families come from nutrients, product/processing, chemistry, HMDB, disease/pathway, text evidence, and KG neighborhoods;
- what one food looks like across recipes;
- how the HPP food-level reference table should be joined to diet logs inside TRE and aggregated to participant-level predictors.

No participant-level TRE data are loaded here.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 140)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

ENHANCED = ROOT / 'outputs' / 'enhanced_hpp'
DOWNSTREAM = ROOT / 'outputs' / 'downstream_features'

SCENARIOS = ['denovo', 'nutrimatch_based']
RECIPES = ['broad_diet_health', 'microbiome', 'mental_health', 'cardiometabolic', 'chemical_metabolomics']

assert DOWNSTREAM.exists(), f'Missing downstream feature folder: {DOWNSTREAM}'
DOWNSTREAM

## Load Downstream Recipe Metadata

Each downstream recipe folder contains:

- `hpp_downstream_feature_table.csv`: one row per HPP food item;
- `hpp_downstream_feature_list.csv`: provenance and role for every column;
- `hpp_downstream_feature_summary.json`: shape and recipe description.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text())

def table_path(scenario, recipe):
    return DOWNSTREAM / scenario / recipe / 'hpp_downstream_feature_table.csv'

def feature_list_path(scenario, recipe):
    return DOWNSTREAM / scenario / recipe / 'hpp_downstream_feature_list.csv'

def summary_path(scenario, recipe):
    return DOWNSTREAM / scenario / recipe / 'hpp_downstream_feature_summary.json'

def load_summary_rows():
    rows = []
    for scenario in SCENARIOS:
        for recipe in RECIPES:
            summary = read_json(summary_path(scenario, recipe))
            rows.append({
                'scenario': scenario,
                'recipe': recipe,
                'hpp_food_rows': summary['hpp_food_rows'],
                'columns': summary['table_shape'][1],
                'identity_columns': summary['identity_columns'],
                'matrix_feature_columns': summary['matrix_feature_columns'],
                'kg_feature_columns': summary['kg_feature_columns'],
                'description': summary['description'],
            })
    return pd.DataFrame(rows)

summary = load_summary_rows()
summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
plot_df = summary.pivot(index='recipe', columns='scenario', values='columns').loc[RECIPES]
plot_df.plot(kind='barh', ax=ax)
ax.set_xlabel('Number of columns in exported HPP food table')
ax.set_title('Downstream feature-table size by recipe')
plt.tight_layout()

## Feature Provenance and Families

For modelling, the key split is not only nutrient vs non-nutrient. We also need to know which columns are direct scenario feature-matrix fields and which are KG-derived neighborhood features.

In [ ]:
def load_feature_list(scenario, recipe):
    return pd.read_csv(feature_list_path(scenario, recipe), low_memory=False)

def feature_family(feature_name):
    name = str(feature_name)
    if name in {
        'hpp_food_id', 'hpp_food_name', 'hpp_product_name', 'hpp_short_description', 'hpp_hebrew_name',
        'hpp_category', 'number_loggings', 'canonical_food_id', 'canonical_name', 'canonical_category',
        'canonical_assignment_method'
    }:
        return 'identity'
    if name.startswith('kg__has_nutrient_amount_per_100g'):
        return 'kg_nutrient_amount'
    if name.startswith('kg__inherits_product_processing') or name.startswith('kg__has_nova') or name.startswith('kg__has_nutriscore'):
        return 'kg_product_processing'
    if name.startswith('kg__linked_to_foodb'):
        return 'kg_chemical_taxonomy'
    if name.startswith('kg__linked_to_hmdb_biospecimen'):
        return 'kg_hmdb_biospecimen'
    if name.startswith('kg__linked_to_hmdb_disease'):
        return 'kg_hmdb_disease'
    if name.startswith('kg__linked_to_hmdb_pathway'):
        return 'kg_hmdb_pathway'
    if name.startswith('canonical_inherited__openfoodfacts') or name.startswith('canonical_inherited__ingredient') or name.startswith('canonical_inherited__additive') or name.startswith('canonical_inherited__nutriscore') or name.startswith('canonical_inherited__nova') or name.startswith('canonical_inherited__pnns') or name.startswith('canonical_inherited__food_groups') or name.startswith('canonical_inherited__brands') or name.startswith('canonical_inherited__labels') or name.startswith('canonical_inherited__allergens'):
        return 'matrix_product_processing'
    if 'foodb' in name.lower() or 'foodatlas_compound' in name.lower() or 'chemical' in name.lower():
        return 'matrix_chemical'
    if 'hmdb' in name.lower() or 'disease' in name.lower() or 'pathway' in name.lower() or 'biospecimen' in name.lower():
        return 'matrix_metabolomics_disease_pathway'
    return 'matrix_nutrient_or_other'

family_rows = []
for scenario in SCENARIOS:
    for recipe in RECIPES:
        fl = load_feature_list(scenario, recipe)
        fl['feature_family'] = fl['feature_name'].map(feature_family)
        counts = fl.groupby(['feature_source', 'feature_family']).size().reset_index(name='n_features')
        counts.insert(0, 'scenario', scenario)
        counts.insert(1, 'recipe', recipe)
        family_rows.append(counts)

feature_family_counts = pd.concat(family_rows, ignore_index=True)
feature_family_counts.sort_values(['scenario', 'recipe', 'feature_source', 'feature_family']).head(80)

In [ ]:
SELECTED_RECIPE = 'broad_diet_health'  # change to microbiome, mental_health, cardiometabolic, chemical_metabolomics

fam = feature_family_counts.query('recipe == @SELECTED_RECIPE')
pivot = fam.pivot_table(index='feature_family', columns='scenario', values='n_features', aggfunc='sum', fill_value=0)
pivot = pivot.sort_values('denovo' if 'denovo' in pivot.columns else pivot.columns[0], ascending=True)
ax = pivot.plot(kind='barh', figsize=(9, 6))
ax.set_title(f'Feature families: {SELECTED_RECIPE}')
ax.set_xlabel('Number of features')
plt.tight_layout()
pivot

## Compare De Novo and NutriMatch-Based Feature Spaces

This section checks which features are shared and which are branch-specific. The main expected difference is nutrient coverage: de novo has more nutrient columns, while non-nutrient KG features are mostly shared because the same Layers 2-5 are applied after the nutrient branch.

In [ ]:
def feature_names(scenario, recipe):
    return set(load_feature_list(scenario, recipe)['feature_name'].astype(str))

comparison_rows = []
for recipe in RECIPES:
    denovo = feature_names('denovo', recipe)
    nm = feature_names('nutrimatch_based', recipe)
    comparison_rows.append({
        'recipe': recipe,
        'denovo_features': len(denovo),
        'nutrimatch_based_features': len(nm),
        'shared_features': len(denovo & nm),
        'denovo_only': len(denovo - nm),
        'nutrimatch_only': len(nm - denovo),
    })
feature_overlap = pd.DataFrame(comparison_rows)
feature_overlap

In [ ]:
# Inspect branch-specific features for the selected recipe.
denovo_only = sorted(feature_names('denovo', SELECTED_RECIPE) - feature_names('nutrimatch_based', SELECTED_RECIPE))
nutrimatch_only = sorted(feature_names('nutrimatch_based', SELECTED_RECIPE) - feature_names('denovo', SELECTED_RECIPE))

print('De novo only:', len(denovo_only))
display(pd.DataFrame({'denovo_only_feature': denovo_only[:80]}))
print('NutriMatch-based only:', len(nutrimatch_only))
display(pd.DataFrame({'nutrimatch_only_feature': nutrimatch_only[:80]}))

## Load One Recipe Table for Modelling Checks

Use `SELECTED_RECIPE` and `SELECTED_SCENARIO` to inspect the table that would be taken into TRE for a given outcome family.

In [ ]:
SELECTED_SCENARIO = 'denovo'  # change to nutrimatch_based
SELECTED_RECIPE = 'broad_diet_health'

feature_table = pd.read_csv(table_path(SELECTED_SCENARIO, SELECTED_RECIPE), low_memory=False, dtype={'hpp_food_id': str})
feature_list = load_feature_list(SELECTED_SCENARIO, SELECTED_RECIPE)
feature_list['feature_family'] = feature_list['feature_name'].map(feature_family)

print(SELECTED_SCENARIO, SELECTED_RECIPE, feature_table.shape)
display(feature_table.head(3))
display(feature_list.groupby(['feature_source', 'feature_family']).size().reset_index(name='n_features'))

In [ ]:
ID_COLS = feature_list.loc[feature_list['feature_source'].eq('hpp_identity'), 'feature_name'].tolist()
MODEL_COLS = [c for c in feature_table.columns if c not in ID_COLS]
NUMERIC_MODEL_COLS = feature_table[MODEL_COLS].select_dtypes(include=[np.number]).columns.tolist()
TEXT_MODEL_COLS = [c for c in MODEL_COLS if c not in NUMERIC_MODEL_COLS]

quality = pd.DataFrame({
    'n_total_columns': [feature_table.shape[1]],
    'n_identity_columns': [len(ID_COLS)],
    'n_model_columns': [len(MODEL_COLS)],
    'n_numeric_model_columns': [len(NUMERIC_MODEL_COLS)],
    'n_text_model_columns': [len(TEXT_MODEL_COLS)],
    'n_hpp_foods': [feature_table['hpp_food_id'].nunique()],
})
quality

In [ ]:
# Missingness and sparsity: useful for paper reporting and model-preprocessing choices.
missing = feature_table[MODEL_COLS].isna().mean().sort_values(ascending=False)
zero_fraction = (feature_table[NUMERIC_MODEL_COLS].fillna(0) == 0).mean().sort_values(ascending=False)

qc = pd.DataFrame({
    'feature': MODEL_COLS,
    'missing_fraction': [missing.get(c, np.nan) for c in MODEL_COLS],
    'zero_fraction_if_numeric': [zero_fraction.get(c, np.nan) for c in MODEL_COLS],
})
qc['feature_family'] = qc['feature'].map(feature_family)

display(qc.groupby('feature_family').agg(
    n_features=('feature', 'count'),
    median_missing=('missing_fraction', 'median'),
    median_zero_fraction=('zero_fraction_if_numeric', 'median'),
).reset_index())

display(qc.sort_values(['missing_fraction', 'zero_fraction_if_numeric'], ascending=False).head(40))

## Inspect a Food Item

This is the practical food-level sanity check: search one HPP food and see which features are non-zero or text-rich. This helps verify whether the extracted task feature table behaves as expected before taking it into TRE.

In [ ]:
QUERY = 'coffee'  # try: milk, chicken, bread, apple, hummus, salad

def search_foods(df, query, max_rows=20):
    query = str(query).lower()
    text_cols = [c for c in ['hpp_food_name', 'hpp_product_name', 'hpp_short_description', 'hpp_category', 'hpp_hebrew_name', 'canonical_name'] if c in df.columns]
    mask = pd.Series(False, index=df.index)
    for col in text_cols:
        mask |= df[col].astype(str).str.lower().str.contains(query, regex=False, na=False)
    show_cols = [c for c in ['hpp_food_id', 'hpp_food_name', 'hpp_product_name', 'hpp_category', 'number_loggings', 'canonical_name'] if c in df.columns]
    return df.loc[mask, show_cols].head(max_rows)

matches = search_foods(feature_table, QUERY)
display(matches)
HPP_FOOD_ID = str(matches.iloc[0]['hpp_food_id']) if not matches.empty else str(feature_table.iloc[0]['hpp_food_id'])
HPP_FOOD_ID

In [ ]:
def selected_food_profile(df, hpp_food_id, top_n=50):
    row = df.query('hpp_food_id == @hpp_food_id').iloc[0]
    numeric = []
    text = []
    for col in MODEL_COLS:
        val = row[col]
        if col in NUMERIC_MODEL_COLS:
            num = pd.to_numeric(pd.Series([val]), errors='coerce').iloc[0]
            if pd.notna(num) and num != 0:
                numeric.append((col, num, feature_family(col)))
        else:
            if pd.notna(val) and str(val).strip() and str(val).lower() != 'nan':
                text.append((col, str(val)[:300], feature_family(col)))
    numeric_df = pd.DataFrame(numeric, columns=['feature', 'value', 'feature_family']).sort_values('value', ascending=False).head(top_n)
    text_df = pd.DataFrame(text, columns=['feature', 'text_preview', 'feature_family']).head(top_n)
    return numeric_df, text_df

selected = feature_table.query('hpp_food_id == @HPP_FOOD_ID')[[c for c in ID_COLS if c in feature_table.columns]].T
numeric_profile, text_profile = selected_food_profile(feature_table, HPP_FOOD_ID)

display(selected)
print('Top non-zero numeric features')
display(numeric_profile)
print('Text evidence fields')
display(text_profile)

## TRE-Side Join and Aggregation Pattern

Inside TRE, the participant diet log should be joined to one of these HPP food-level reference tables by `hpp_food_id` / `food_id`.

For NutriMatch-style prediction tasks, the food-level features become participant-level predictors after multiplying dose-scalable per-100 g nutrient features by consumed grams and aggregating across diet records. Annotation and KG-neighborhood features are usually aggregated as exposure indicators, counts, or amount-weighted evidence scores.

The code below is a template. It uses a tiny fake diet log only to show the mechanics.

In [ ]:
# Example only. In TRE, replace this with the real participant-level HPP diet log.
example_diet_log = pd.DataFrame({
    'participant_id': ['p1', 'p1', 'p2'],
    'food_id': [HPP_FOOD_ID, HPP_FOOD_ID, feature_table.iloc[1]['hpp_food_id']],
    'grams_consumed': [50, 120, 80],
    'diet_date': pd.to_datetime(['2024-01-01', '2024-01-02', '2024-01-01']),
})

# Minimal join table: remove human-readable descriptors if the model should not see labels.
join_table = feature_table.rename(columns={'hpp_food_id': 'food_id'}).copy()
joined = example_diet_log.merge(join_table, on='food_id', how='left', validate='many_to_one')
joined[['participant_id', 'food_id', 'grams_consumed', 'hpp_food_name', 'canonical_name']].head()

In [ ]:
# Recommended event-level transform.
# Nutrient amount columns and kg nutrient amount columns are dose-scalable.
# Other numeric annotation features are kept as exposure evidence and can be amount-weighted or presence/count aggregated.

def is_dose_scalable_feature(col):
    if col.startswith('kg__has_nutrient_amount_per_100g'):
        return True
    if col in feature_list['feature_name'].values:
        family = feature_family(col)
        return family == 'matrix_nutrient_or_other' and not col.startswith('canonical_inherited__')
    return False

model_numeric = [c for c in NUMERIC_MODEL_COLS if c in joined.columns]
dose_scalable_cols = [c for c in model_numeric if is_dose_scalable_feature(c)]
annotation_numeric_cols = [c for c in model_numeric if c not in dose_scalable_cols]

for col in dose_scalable_cols:
    joined[col + '__amount'] = pd.to_numeric(joined[col], errors='coerce') * joined['grams_consumed'] / 100.0

for col in annotation_numeric_cols:
    joined[col + '__exposure_weighted'] = pd.to_numeric(joined[col], errors='coerce').fillna(0) * joined['grams_consumed'] / 100.0

amount_cols = [c for c in joined.columns if c.endswith('__amount')]
exposure_cols = [c for c in joined.columns if c.endswith('__exposure_weighted')]

participant_features = joined.groupby('participant_id')[amount_cols + exposure_cols].sum(min_count=1).reset_index()
print('Participant-level predictor table shape:', participant_features.shape)
participant_features.head()

## Model-Ready Export Notes

Before modelling inside TRE:

1. Choose one scenario: `denovo` or `nutrimatch_based`.
2. Choose one recipe based on outcome family.
3. Join by HPP `food_id`.
4. Transform per-100 g nutrient values using consumed grams.
5. Aggregate diet events to the participant/time-window level required by the outcome.
6. Keep the feature list file with the model output for reproducibility.

For a paper comparison, the clean first analysis is to run the same prediction task twice: once with `denovo` features and once with `nutrimatch_based` features, keeping the recipe, model family, cross-validation, and outcome definition identical.

In [ ]:
# Compact paper-reporting table for all currently exported recipes.
report = summary.merge(feature_overlap, on='recipe', how='left')
report_cols = [
    'scenario', 'recipe', 'hpp_food_rows', 'columns', 'matrix_feature_columns', 'kg_feature_columns',
    'shared_features', 'denovo_only', 'nutrimatch_only', 'description'
]
report = report[report_cols]
report

In [ ]:
# Optional: save a compact overview table for manuscript/results drafting.
out = ENHANCED / 'downstream_recipe_overview_for_paper.csv'
report.to_csv(out, index=False)
out